# 北京数学分班考试题提取器

Extract questions from Beijing Math Placement Exam PDF using Gemini

In [ ]:
!pip install -q google-generativeai PyMuPDF pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuration
PROJECT_PATH = '/content/drive/MyDrive/project/ai_math'
PDF_PATH = f'{PROJECT_PATH}/data/北京各区分班考试真题集—数学 .pdf'
OUTPUT_JSON = f'{PROJECT_PATH}/output/beijing_math_questions.json'
IMAGES_FOLDER = f'{PROJECT_PATH}/output/beijing_math_images'

import os
os.makedirs(IMAGES_FOLDER, exist_ok=True)

In [ ]:
import google.generativeai as genai
from google.colab import userdata
import json
import fitz
from PIL import Image

# Setup Gemini
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except:
    GEMINI_API_KEY = input('Enter Gemini API key: ')

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')
print("✅ Gemini ready")

In [ ]:
# Parsing prompt for Chinese math questions
CHINESE_MATH_PROMPT = """
这是一页北京数学分班考试题。请仔细分析并提取所有信息。

提取内容：
1. 页面标题/章节名称（如果有）
2. 所有题目的完整内容
3. 每道题的题号
4. 题目类型（选择题、填空题、解答题等）
5. 涉及的知识点

返回JSON格式：
{
  "page_title": "页面标题",
  "questions": [
    {
      "number": "1",
      "type": "选择题/填空题/解答题",
      "content": "题目完整内容，包括所有文字和数学表达式",
      "options": ["A. ...", "B. ..."],  // 如果是选择题
      "knowledge_points": ["知识点1", "知识点2"]
    }
  ]
}

重要：
- 保留所有数学符号和公式
- 如果题目跨页，标注"(续)"
- 如果看不清，标注"[unclear]"
- 只返回JSON，不要其他文字
"""

In [ ]:
def extract_page_image(pdf_path, page_num, output_folder):
    """Extract page as image"""
    doc = fitz.open(pdf_path)
    page = doc[page_num - 1]
    mat = fitz.Matrix(2, 2)
    pix = page.get_pixmap(matrix=mat)
    img_path = f"{output_folder}/page_{page_num}.jpg"
    pix.save(img_path)
    doc.close()
    return img_path

def parse_page_with_gemini(image_path, page_num):
    """Parse Chinese math questions with Gemini"""
    try:
        img = Image.open(image_path)
        response = model.generate_content([CHINESE_MATH_PROMPT, img])
        
        text = response.text.strip()
        if text.startswith('```json'):
            text = text[7:]
        if text.startswith('```'):
            text = text[3:]
        if text.endswith('```'):
            text = text[:-3]
        text = text.strip()
        
        data = json.loads(text)
        data['page'] = page_num
        return data
    except Exception as e:
        print(f"❌ Error on page {page_num}: {e}")
        return None

In [ ]:
def process_pages(start_page, end_page):
    """Process a range of pages"""
    results = []
    
    for page_num in range(start_page, end_page + 1):
        print(f"\n📄 Processing page {page_num}...")
        
        # Extract image
        img_path = extract_page_image(PDF_PATH, page_num, IMAGES_FOLDER)
        print(f"  ✅ Image: {img_path}")
        
        # Parse with Gemini
        data = parse_page_with_gemini(img_path, page_num)
        
        if data:
            results.append(data)
            num_questions = len(data.get('questions', []))
            print(f"  ✅ Extracted {num_questions} questions")
            if data.get('page_title'):
                print(f"  📝 Title: {data['page_title']}")
        else:
            print(f"  ❌ Failed")
    
    return results

## Test with Sample Pages

In [ ]:
# Test with pages 5-7 (skip cover pages)
test_results = process_pages(5, 7)

# Display results
for result in test_results:
    print(f"\n{'='*60}")
    print(json.dumps(result, ensure_ascii=False, indent=2)[:500])
    print("...")

## Process All Pages

In [ ]:
# Process in batches
all_results = []

# Skip first 4 pages (cover/intro), process pages 5-256
for batch_start in range(5, 257, 10):
    batch_end = min(batch_start + 9, 256)
    print(f"\n{'='*60}")
    print(f"Batch: pages {batch_start}-{batch_end}")
    print(f"{'='*60}")
    
    batch_results = process_pages(batch_start, batch_end)
    all_results.extend(batch_results)
    
    # Save after each batch
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"\n💾 Saved {len(all_results)} pages so far")
    
    import time
    time.sleep(2)

print(f"\n✅ Complete! Total pages: {len(all_results)}")

## View Results

In [ ]:
# Load and analyze results
with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
    data = json.load(f)

total_questions = sum(len(p.get('questions', [])) for p in data)
print(f"📊 Statistics:")
print(f"  Pages processed: {len(data)}")
print(f"  Total questions: {total_questions}")
print(f"  Avg questions/page: {total_questions/len(data):.1f}")

# Show sample
if data:
    print(f"\n📄 Sample page {data[0]['page']}:")
    print(json.dumps(data[0], ensure_ascii=False, indent=2)[:800])

In [ ]:
# Download results
from google.colab import files
files.download(OUTPUT_JSON)